# YOLOv8 - Entrenamiento para FFAI Assistant

Este notebook entrena YOLOv8 para detectar elementos en Free Fire.

## Clases a detectar:
- 0: enemy (enemigo)
- 1: teammate (aliado)
- 2: loot (botín)
- 3: weapon (arma)
- 4: vehicle (vehículo)
- 5: zone (zona segura/peligrosa)

In [ ]:
# Verificar GPU
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# Instalar dependencias
!pip install -q ultralytics
!pip install -q tensorflow

In [ ]:
# Importar librerías
from ultralytics import YOLO
import os
import yaml

## 1. Cargar Dataset

In [ ]:
# Opción A: Usar modelo pre-entrenado COCO (para empezar rápido)
# Detecta 'person' que podemos usar como base para enemigos

# Opción B: Dataset personalizado (recomendado para producción)
# Descomenta si tienes dataset en Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# dataset_path = '/content/drive/MyDrive/FFAI/dataset'

## 2. Cargar Modelo Base

In [ ]:
# Cargar YOLOv8n (nano) - más rápido para móvil
# Opciones: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
model = YOLO('yolov8n.pt')

print(f"Modelo cargado: {model.info()}")

## 3. Entrenar (opcional si usas pre-entrenado)

In [ ]:
# Solo si tienes dataset personalizado
# Descomenta para entrenar:

# results = model.train(
#     data=f'{dataset_path}/data.yaml',
#     epochs=100,
#     imgsz=640,
#     batch=16,
#     device='cuda',
#     workers=8,
#     patience=20,
#     save=True,
#     project='ffai_training',
#     name='yolov8n_ffai'
# )

## 4. Exportar a TFLite

In [ ]:
# Exportar a TFLite con optimizaciones para móvil
# FP16 = Half precision (reduce tamaño 50%)

model.export(
    format='tflite',
    int8=False,        # No cuantizar a INT8 (pierde mucha precisión)
    half=True,         # FP16 = buen balance precisión/tamaño
    nms=True,          # Incluir Non-Maximum Suppression
    simplify=True,     # Simplificar grafo
    workspace=2,       # GB para optimizador
    batch=1            # Batch size para inferencia
)

print("Exportación completada!")

In [ ]:
# Verificar archivo generado
import os

tflite_path = '/content/yolov8n_fp16.tflite'
if os.path.exists(tflite_path):
    size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f"✓ Modelo generado: {tflite_path}")
    print(f"✓ Tamaño: {size_mb:.2f} MB")
else:
    # Buscar en directorio de resultados si entrenamos
    import glob
    tflite_files = glob.glob('/content/**/*.tflite', recursive=True)
    if tflite_files:
        for f in tflite_files:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"✓ Encontrado: {f} ({size_mb:.2f} MB)")
            tflite_path = f

## 5. Descargar Modelo

In [ ]:
# Descargar archivo
from google.colab import files

# Renombrar al formato esperado por la app
final_name = 'yolov8n_fp16.tflite'
if tflite_path != f'/content/{final_name}':
    os.rename(tflite_path, f'/content/{final_name}')

files.download(f'/content/{final_name}')
print(f"\n✓ Descargando {final_name}...")

## 6. Instrucciones Post-Descarga

1. El archivo descargado debe copiarse a:
   ```
   app/src/main/assets/yolov8n_fp16.tflite
   ```

2. En Android Studio, sincronizar proyecto

3. Verificar que `aaptOptions` está configurado en `build.gradle`:
   ```gradle
   aaptOptions {
       noCompress 'tflite'
   }
   ```

## Notas de Optimización

- **FP16**: Reduce tamaño 50%, buena precisión
- **INT8**: Reduce tamaño 75%, puede perder precisión en detección de enemigos pequeños
- **NMS integrado**: Incluir en modelo evita procesamiento post-inferencia
- **Input size 640**: Balance entre velocidad y precisión para móvil